# Text-to-Image Generation with ERNIE-Image-Turbo and OpenVINO

[ERNIE-Image-Turbo](https://huggingface.co/Baidu/ERNIE-Image-Turbo) is Baidu's production-ready, open-source image generation model based on the ERNIE family.

**Highlights**

- High-quality photorealistic image generation with strong bilingual (Chinese & English) support
- Uses Diffusion Transformer architecture with Mistral3 text encoder
- Optional **Prompt Enhancer (PE)** — a built-in language model that automatically expands short prompts into detailed visual descriptions
- Fast 8-step generation with flow matching scheduler

More details about the model can be found in the [model card](https://huggingface.co/Baidu/ERNIE-Image-Turbo).

In this tutorial we consider how to convert and optimize ERNIE-Image-Turbo model using OpenVINO.

> **Note**: This notebook requires at least 32 GB RAM for model conversion and ~16 GB for INT4 inference.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Convert model with OpenVINO](#Convert-model-with-OpenVINO)
  - [Select weight format](#Select-weight-format)
  - [Export model using Optimum Intel](#Export-model-using-Optimum-Intel)
- [Run OpenVINO model inference](#Run-OpenVINO-model-inference)
- [Interactive demo](#Interactive-demo)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/ernie-image/ernie-image.ipynb" />

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

In [ ]:
import platform
import requests
from pathlib import Path

if not Path("cmd_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py")
    open("cmd_helper.py", "w").write(r.text)

if not Path("notebook_utils.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py")
    open("notebook_utils.py", "w").write(r.text)

if not Path("pip_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/pip_helper.py")
    open("pip_helper.py", "w").write(r.text)

from pip_helper import pip_install

%pip uninstall -q -y diffusers optimum-intel

pip_install(
    "-q",
    "gradio>=4.19,<6",
    "torch>=2.8",
    "nncf>=2.15.0",
    "accelerate",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cpu",
)
pip_install("-q", "git+https://github.com/huggingface/diffusers.git@6a339ce637db184c2e1a10ec90ac0e292beb76ac")
pip_install("-q", "git+https://github.com/openvino-dev-samples/optimum-intel.git@5cf0d7c88fe57ed014c432c9eac26f299557e8ed")
pip_install("-qU", "openvino>=2025.4")

if platform.system() == "Darwin":
    pip_install("numpy<2.0.0")

from notebook_utils import collect_telemetry

collect_telemetry("ernie-image.ipynb")

## Convert model with OpenVINO
[back to top ⬆️](#Table-of-contents:)

ERNIE-Image-Turbo uses a Diffusion Transformer (DiT) architecture. The pipeline consists of several key components:

* **Text Encoder** — Mistral3 language model that creates conditioning embeddings from text prompts.
* **Transformer** — ErnieImageTransformer2DModel for step-by-step denoising of the latent image representation.
* **VAE** — AutoencoderKLFlux2 for encoding/decoding between pixel and latent space.
* **PE (Prompt Enhancer)** — *(Optional)* A Ministral3 causal language model that automatically expands short prompts into detailed visual descriptions.

We use [Optimum Intel](https://huggingface.co/docs/optimum/intel/index) to export all components to OpenVINO IR format.

### Select weight format
[back to top ⬆️](#Table-of-contents:)

INT4 weight compression significantly reduces model size and improves inference speed with minimal quality loss.

In [ ]:
import ipywidgets as widgets

model_id = "Baidu/ERNIE-Image-Turbo"

to_compress = widgets.Checkbox(
    value=True,
    description="INT4 weight compression",
    disabled=False,
)

load_pe = widgets.Checkbox(
    value=False,
    description="Load Prompt Enhancer (PE)",
    disabled=False,
)

widgets.VBox([to_compress, load_pe])

### Export model using Optimum Intel
[back to top ⬆️](#Table-of-contents:)

[Optimum Intel](https://huggingface.co/docs/optimum/intel/index) provides `optimum-cli` for model export. The PE model (if present in the source) is automatically detected and exported alongside the main pipeline components.

In [ ]:
from pathlib import Path

model_base_dir = Path(model_id.split("/")[-1])
additional_args = {"task": "text-to-image"}

if to_compress.value:
    model_dir = model_base_dir / "INT4"
    additional_args.update({"weight-format": "int4", "ratio": "0.8"})
else:
    model_dir = model_base_dir / "FP16"
    additional_args.update({"weight-format": "fp16"})

print(f"Model will be exported to: {model_dir}")
print(f"Weight format: {'INT4' if to_compress.value else 'FP16'}")
print(f"PE model: {'will be exported' if load_pe.value else 'skipped'}")

In [ ]:
from cmd_helper import optimum_cli

if not model_dir.exists():
    optimum_cli(model_id, model_dir, additional_args=additional_args)
else:
    print(f"Model already exists at {model_dir}, skipping export.")

## Run OpenVINO model inference
[back to top ⬆️](#Table-of-contents:)

Select the device for running inference using OpenVINO.

In [ ]:
from notebook_utils import device_widget

device = device_widget(default="CPU", exclude=["NPU"])
device

In [ ]:
import ipywidgets as widgets

model_available = (model_base_dir / "INT4").is_dir()
use_quantized_models = widgets.Checkbox(
    value=model_available,
    description="Use compressed models",
    disabled=not model_available,
)

use_quantized_models

`OVErnieImagePipeline` provides a ready-to-use API fully compatible with the diffusers `ErnieImagePipeline`. The `load_pe` parameter controls whether the Prompt Enhancer model is loaded.

In [ ]:
from optimum.intel import OVErnieImagePipeline
import torch

model_dir = model_base_dir / "INT4" if use_quantized_models.value else model_base_dir / "FP16"

ov_pipe = OVErnieImagePipeline.from_pretrained(
    model_dir,
    device=device.value,
    load_pe=load_pe.value,
)
print(f"Pipeline loaded from {model_dir}")
print(f"PE model loaded: {ov_pipe.pe is not None}")

Now let's generate an image. ERNIE-Image-Turbo supports both Chinese and English prompts.

In [ ]:
prompt = "A cute cat sitting on a colorful cushion, studio lighting, high quality, detailed fur texture"

generator = torch.Generator("cpu").manual_seed(42)

result = ov_pipe(
    prompt=prompt,
    height=512,
    width=512,
    num_inference_steps=8,
    guidance_scale=1.0,
    generator=generator,
)

result.images[0]

If the PE model was loaded, you can use `use_pe=True` to automatically enhance the prompt before generation:

In [ ]:
if ov_pipe.pe is not None:
    generator = torch.Generator("cpu").manual_seed(42)

    result_pe = ov_pipe(
        prompt="a cute cat",
        height=512,
        width=512,
        num_inference_steps=8,
        guidance_scale=1.0,
        generator=generator,
        use_pe=True,
    )

    print(f"Enhanced prompt: {result_pe.revised_prompts[0][:200]}...")
    display(result_pe.images[0])
else:
    print("PE model not loaded. Set load_pe=True in the widget above and re-run to enable Prompt Enhancer.")

## Interactive demo
[back to top ⬆️](#Table-of-contents:)

In [ ]:
from gradio_helper import make_demo

demo = make_demo(ov_pipe, enable_pe=(ov_pipe.pe is not None))

# if you are launching remotely, specify server_name and server_port
#  demo.launch(server_name='your server name', server_port='server port in int')
# if you have any issue to launch on your platform, you can pass share=True to launch method:
# demo.launch(share=True)
# it creates a publicly shareable link for the interface. Read more in the docs: https://gradio.app/docs/
try:
    demo.launch(debug=True)
except Exception:
    demo.launch(debug=True, share=True)